# VentriQ — Phase 5: Smartphone Photograph Simulation & Vision Preprocessing

**Objective:** Simulate realistic real-world smartphone camera photographs of printed ECG recordings, and implement the computer vision preprocessing pipeline that detects, rectifies, and prepares images for MobileNetV3 inference.

### Complete Vision Pipeline Architecture:
1. **Raw 12-Lead ECG Waveform** $\rightarrow$ **Phase 4 Canonical Paper ECG** ($280\text{ mm} \times 190\text{ mm}$ @ 150 DPI)
2. **Smartphone Camera Simulation (`SmartphoneECGSimulator`):** 3D camera pose (yaw, pitch, roll homography), radial illumination falloff, cast shadows (hand/phone body), specular glare, page curl, defocus blur, sensor noise, and JPEG compression.
3. **Document Boundary & Corner Localization (`detect_document_corners`):** Otsu bimodal segmentation, Sobel gradient edge analysis, and extremal quadrilateral corner fitting.
4. **Perspective Homography Unwarping (`rectify_document`):** Four-point perspective transform restoring canonical rectilinear geometry.
5. **Illumination Correction & Contrast Enhancement (`ECGImageEnhancer`):** Background illumination division normalization and CLAHE local adaptive equalization.
6. **MobileNetV3 Tensor Formatting (`image_to_mobilenet_tensor`):** Standardized $3 \times 224 \times 224$ PyTorch float32 tensor with ImageNet channel standardization.

In [ ]:
from pathlib import Path
import sys

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader

from src.data.dataset import VentriQTorchDataset
from src.preprocessing.image import ECGImageEnhancer, normalize_illumination, apply_clahe
from src.preprocessing.perspective import detect_document_corners, rectify_document, DocumentRectifier
from src.preprocessing.transforms import MobileNetV3Preprocessor, image_to_mobilenet_tensor, extract_12_lead_crops
from src.rendering.augmentation import DistortionConfig, SmartphoneECGSimulator
from src.rendering.layout import ECGLayoutConfig
from src.rendering.renderer import ECGPaperRenderer
from src.utils.paths import get_raw_echonext_dir, REPORTS_DIR, FIGURES_DIR

print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")

## 1. Load Real Record and Render Ground-Truth Paper ECG

In [ ]:
dataset = VentriQTorchDataset(root=get_raw_echonext_dir())
sample = dataset[0]

renderer = ECGPaperRenderer(config=ECGLayoutConfig(dpi=150))
canonical_ecg = renderer.render(sample["waveform"], record=sample)
canonical_pil = canonical_ecg.pil_image

print(f"Record ID: {sample['record_id']} | LVEF: {sample['lvef'].item():.1f}%")
print(f"Canonical Paper Dimensions: {canonical_pil.size} (W x H)")

plt.figure(figsize=(12, 7), dpi=100)
plt.imshow(canonical_pil)
plt.axis("off")
plt.title(f"Phase 4 Canonical Paper Ground Truth: {sample['record_id']}", fontsize=11)
plt.show()

## 2. Simulate Realistic Smartphone Photo Capture

In [ ]:
dist_cfg = DistortionConfig.from_preset("moderate")
simulator = SmartphoneECGSimulator(config=dist_cfg, seed=42)

sim_photo = simulator.simulate(canonical_pil)
photo_pil = sim_photo.image

print(f"Simulated Photo Dimensions: {photo_pil.size}")
print(f"Camera Yaw:   {sim_photo.metadata['camera_yaw_deg']:.1f}°")
print(f"Camera Pitch: {sim_photo.metadata['camera_pitch_deg']:.1f}°")
print(f"Camera Roll:  {sim_photo.metadata['camera_roll_deg']:.1f}°")
print(f"Ground-Truth Transformed Corners:\n{sim_photo.corners_transformed}")

plt.figure(figsize=(12, 8), dpi=100)
plt.imshow(photo_pil)
plt.scatter(sim_photo.corners_transformed[:, 0], sim_photo.corners_transformed[:, 1], color="red", s=40, zorder=5, label="Ground-Truth Corners")
plt.legend(loc="upper right")
plt.axis("off")
plt.title("Simulated Smartphone Photograph (3D Tilt, Hand Shadow, Vignetting, Blur & Noise)", fontsize=11)
plt.show()

## 3. Automated Document Boundary Localization & Corner Detection

In [ ]:
doc = detect_document_corners(photo_pil)

print(f"Confidence Score:     {doc.confidence * 100:.1f}%")
print(f"Estimated Aspect:     {doc.aspect_ratio:.3f} (Standard: 1.474)")
print(f"Detected Corners:\n{doc.corners}")

# Measure error against simulator's ground-truth corners
gt = sim_photo.corners_transformed
corner_errors = [np.linalg.norm(g - d) for g, d in zip(gt, doc.corners)]
print(f"Mean Corner Error:    {np.mean(corner_errors):.2f} pixels")

plt.figure(figsize=(12, 8), dpi=100)
plt.imshow(photo_pil)
plt.scatter(gt[:, 0], gt[:, 1], color="red", s=60, marker="o", label="Ground Truth")
plt.scatter(doc.corners[:, 0], doc.corners[:, 1], color="lime", s=40, marker="x", label="Detected Corners")
plt.legend(loc="upper right")
plt.axis("off")
plt.title(f"Corner Localization Overlay (Mean Error: {np.mean(corner_errors):.2f} px)", fontsize=11)
plt.show()

## 4. Four-Point Homography Perspective Rectification

In [ ]:
rectified = rectify_document(photo_pil, corners=doc, target_size=canonical_pil.size)
rectified_pil = rectified.image

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
axes[0].imshow(photo_pil)
axes[0].axis("off")
axes[0].set_title("Before Rectification: Tilted Smartphone Photo", fontsize=10)

axes[1].imshow(rectified_pil)
axes[1].axis("off")
axes[1].set_title("After Rectification: Canonical Rectilinear Canvas", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Illumination Correction & Contrast Enhancement (CLAHE)

In [ ]:
enhancer = ECGImageEnhancer(illumination_sigma=30.0, enable_clahe=True, clahe_clip_limit=2.5)
enhanced_pil = enhancer(rectified_pil)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
axes[0].imshow(rectified_pil)
axes[0].axis("off")
axes[0].set_title("Rectified with Residual Shadows", fontsize=10)

axes[1].imshow(enhanced_pil)
axes[1].axis("off")
axes[1].set_title("Illumination Division + CLAHE Enhanced", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Model-Ready MobileNetV3 Tensor Formatting

In [ ]:
# Direct MobileNetV3 tensor formatting (3, 224, 224)
tensor_224 = image_to_mobilenet_tensor(enhanced_pil, target_size=(224, 224), normalize_imagenet=True)
print(f"MobileNetV3 Tensor Shape: {tensor_224.shape} (C x H x W)")
print(f"Tensor Value Range:        [{tensor_224.min():.2f}, {tensor_224.max():.2f}]")

# Multi-crop 12-lead patches tensor (12, 3, 112, 224)
lead_crops_tensor = extract_12_lead_crops(enhanced_pil, patch_size=(224, 112), normalize_imagenet=True)
print(f"12-Lead Crop Tensor Shape: {lead_crops_tensor.shape} (12 Leads x C x H x W)")

# Visualizing MobileNetV3 standardized input
mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
vis_tensor = np.clip((tensor_224.numpy() * std + mean).transpose(1, 2, 0), 0.0, 1.0)

plt.figure(figsize=(6, 6), dpi=100)
plt.imshow(vis_tensor)
plt.axis("off")
plt.title("MobileNetV3 Model Input (224 x 224)", fontsize=10)
plt.show()

print("\nPhase 5 Vision Preprocessor is verified and model-ready!")